# 10 - Silver Events

Normalizes UTC timestamps and devices, quarantines invalid shapes, deduplicates by newest ingest time, and enables CDF.

In [ ]:
%run ./05_bronze_common

In [ ]:
import uuid
from datetime import datetime, timezone

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window


def run_version(run_id: str) -> int:
    return int(run_id.replace("-", "")[:15], 16)


def merge_quarantine(table_name: str, errors) -> int:
    count = errors.count()
    if count:
        (
            DeltaTable.forName(spark, table_name)
            .alias("target")
            .merge(errors.alias("source"), "target.error_id = source.error_id")
            .whenNotMatchedInsertAll()
            .execute()
        )
    return count


def silver_dq(
    catalog: str,
    source_name: str,
    pipeline_run_id: str,
    valid_count: int,
    errors,
    duplicate_count: int = 0,
) -> None:
    error_counts = errors.groupBy("error_code").count().collect()
    if duplicate_count:
        error_counts.append(Row(error_code="duplicate_key", count=duplicate_count))
    now = datetime.now(timezone.utc)
    rows = [
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="valid",
            error_code=None,
            record_count=int(valid_count),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
    ]
    rows.extend(
        Row(
            quality_date=now.date(),
            source_name=source_name,
            pipeline_stage="silver",
            quality_status="error",
            error_code=row.error_code,
            record_count=int(row["count"]),
            pipeline_run_id=pipeline_run_id,
            recorded_ts=now,
        )
        for row in error_counts
    )
    append_delta(
        spark.createDataFrame(rows, DQ_SCHEMA),
        f"{catalog}.control.data_quality_results",
        f"{source_name}_silver_dq",
        run_version(pipeline_run_id),
    )


def silver_error(
    error_code,
    error_reason,
    record_key,
    raw_payload,
    rescued_data,
    source_file,
    batch_id,
    load_date,
    ingest_ts,
    pipeline_run_id,
):
    return [
        F.sha2(
            F.concat_ws(
                "||",
                source_file,
                F.coalesce(record_key, F.lit("")),
                raw_payload,
                error_code,
            ),
            256,
        ).alias("error_id"),
        F.lit("silver").alias("error_stage"),
        error_code.alias("error_code"),
        error_reason.alias("error_reason"),
        record_key.alias("record_key"),
        raw_payload.alias("raw_payload"),
        rescued_data.alias("rescued_data"),
        source_file.alias("source_file"),
        batch_id.alias("batch_id"),
        load_date.alias("load_date"),
        ingest_ts.alias("ingest_ts"),
        F.current_date().alias("quarantine_date"),
        pipeline_run_id.alias("pipeline_run_id"),
    ]

In [ ]:
dbutils.widgets.text("catalog", "abc_retail", "Target catalog")
dbutils.widgets.text("job_run_id", "manual", "Notebook or job run ID")
catalog = validate_identifier(dbutils.widgets.get("catalog").strip(), "catalog")
job_run_id = dbutils.widgets.get("job_run_id").strip()
spark.conf.set("spark.sql.session.timeZone", "UTC")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog}`.`silver`")

pipeline_name = "silver_events"
source_name = "clickstream_events"
pipeline_run_id = str(uuid.uuid4())
start_ts = datetime.now(timezone.utc)
bronze_table = f"{catalog}.bronze.bronze_clickstream"
target_table = f"{catalog}.silver.silver_events"
quarantine_table = f"{catalog}.quarantine.clickstream_errors"

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS `{catalog}`.`silver`.`silver_events` (
        event_id STRING NOT NULL,
        event_time TIMESTAMP NOT NULL,
        event_date DATE NOT NULL,
        source_ingest_time TIMESTAMP,
        user_id STRING,
        session_id STRING,
        event_type STRING,
        page STRING,
        device STRUCT<os: STRING, model: STRING>,
        attributes MAP<STRING, STRING>,
        ingest_ts TIMESTAMP NOT NULL,
        source_file STRING NOT NULL,
        batch_id STRING NOT NULL,
        load_date DATE NOT NULL,
        pipeline_run_id STRING NOT NULL
    )
    USING DELTA
    PARTITIONED BY (event_date)
    TBLPROPERTIES (
        'delta.enableChangeDataFeed' = 'true',
        'comment' = 'Deduplicated and conformed clickstream events'
    )
    """
)

upsert_audit(
    catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts,
    "RUNNING", [bronze_table], job_run_id
)

try:
    bronze = spark.table(bronze_table)
    device_raw = F.coalesce(
        F.get_json_object("raw_payload", "$.device"),
        F.get_json_object("_rescued_data", "$.device"),
    )
    parsed = (
        bronze
        .withColumn("event_id_parsed", F.get_json_object("raw_payload", "$.event_id"))
        .withColumn("event_time_parsed", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.event_time'))"))
        .withColumn("source_ingest_time", F.expr("try_to_timestamp(get_json_object(raw_payload, '$.ingest_time'))"))
        .withColumn("user_id_parsed", F.get_json_object("raw_payload", "$.user_id"))
        .withColumn("session_id_parsed", F.get_json_object("raw_payload", "$.session_id"))
        .withColumn("event_type_parsed", F.get_json_object("raw_payload", "$.event_type"))
        .withColumn("page_parsed", F.get_json_object("raw_payload", "$.page"))
        .withColumn("device_raw", device_raw)
        .withColumn(
            "device_os",
            F.when(F.col("device_raw").startswith("{"), F.get_json_object("device_raw", "$.os"))
            .when(F.col("device_raw").contains("|"), F.split("device_raw", r"\|").getItem(0)),
        )
        .withColumn(
            "device_model",
            F.when(F.col("device_raw").startswith("{"), F.get_json_object("device_raw", "$.model"))
            .when(F.col("device_raw").contains("|"), F.split("device_raw", r"\|").getItem(1)),
        )
        .withColumn(
            "attributes_parsed",
            F.from_json(F.get_json_object("raw_payload", "$.attributes"), "map<string,string>"),
        )
        .withColumn(
            "error_code",
            F.when(F.col("event_id_parsed").isNull(), F.lit("missing_key"))
            .when(F.col("event_time_parsed").isNull(), F.lit("invalid_timestamp"))
            .when(
                F.col("device_os").isNull() | F.col("device_model").isNull(),
                F.lit("schema_mismatch"),
            ),
        )
        .withColumn(
            "error_reason",
            F.when(F.col("error_code") == "missing_key", F.lit("event_id is null"))
            .when(F.col("error_code") == "invalid_timestamp", F.lit("event_time cannot be parsed"))
            .when(F.col("error_code") == "schema_mismatch", F.lit("device cannot be normalized to {os, model}")),
        )
    )

    errors = parsed.filter(F.col("error_code").isNotNull()).select(
        *silver_error(
            F.col("error_code"), F.col("error_reason"), F.col("event_id_parsed"),
            F.col("raw_payload"), F.col("_rescued_data"), F.col("source_file"),
            F.col("batch_id"), F.col("load_date"), F.col("ingest_ts"),
            F.lit(pipeline_run_id),
        )
    )
    error_count = merge_quarantine(quarantine_table, errors)

    candidates = parsed.filter(F.col("error_code").isNull()).select(
        F.col("event_id_parsed").alias("event_id"),
        F.col("event_time_parsed").alias("event_time"),
        F.to_date("event_time_parsed").alias("event_date"),
        "source_ingest_time",
        F.col("user_id_parsed").alias("user_id"),
        F.col("session_id_parsed").alias("session_id"),
        F.col("event_type_parsed").alias("event_type"),
        F.col("page_parsed").alias("page"),
        F.struct(F.col("device_os").alias("os"), F.col("device_model").alias("model")).alias("device"),
        F.col("attributes_parsed").alias("attributes"),
        "ingest_ts", "source_file", "batch_id", "load_date",
        F.lit(pipeline_run_id).alias("pipeline_run_id"),
    )
    ranking = Window.partitionBy("event_id").orderBy(
        F.col("source_ingest_time").desc_nulls_last(),
        F.col("ingest_ts").desc(),
        F.col("source_file").desc(),
    )
    ranked = candidates.withColumn("_rank", F.row_number().over(ranking))
    duplicate_count = ranked.filter(F.col("_rank") > 1).count()
    latest = ranked.filter(F.col("_rank") == 1).drop("_rank")

    target = DeltaTable.forName(spark, target_table)
    (
        target.alias("target")
        .merge(latest.alias("source"), "target.event_id = source.event_id")
        .whenMatchedUpdate(
            condition="""
                coalesce(source.source_ingest_time, source.ingest_ts) > coalesce(target.source_ingest_time, target.ingest_ts)
                OR (
                    coalesce(source.source_ingest_time, source.ingest_ts) = coalesce(target.source_ingest_time, target.ingest_ts)
                    AND source.ingest_ts > target.ingest_ts
                )
                OR (
                    coalesce(source.source_ingest_time, source.ingest_ts) = coalesce(target.source_ingest_time, target.ingest_ts)
                    AND source.ingest_ts = target.ingest_ts
                    AND source.source_file > target.source_file
                )
            """,
            set={column: f"source.{column}" for column in latest.columns},
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    valid_count = latest.count()
    silver_dq(catalog, source_name, pipeline_run_id, valid_count, errors, duplicate_count)
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts,
        "SUCCESS", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc),
        rows_read=bronze.count(), rows_written=valid_count,
        rows_quarantined=error_count + duplicate_count,
    )
except Exception as exc:
    upsert_audit(
        catalog, pipeline_run_id, pipeline_name, "full_bronze_merge", start_ts,
        "FAILED", [bronze_table], job_run_id, end_ts=datetime.now(timezone.utc),
        error_message=str(exc)[:4000],
    )
    raise